In [29]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import re
from scipy.stats import pearsonr

In [30]:
year = 2024
csv_localias = f"final_data/home_priority_{year}.csv"
bracket_real_csv = f"final_data/bracket_real_{year}.csv"

In [31]:
def get_home_priority(csv_path, team1, team2):
    df = pd.read_csv(csv_path)
    team1 = team1.upper()
    team2 = team2.upper()

    row1 = df[df['team'] == team1]
    row2 = df[df['team'] == team2]

    if row1.empty or row2.empty:
        raise ValueError(f"One or both teams isn't in the archive: {team1}, {team2}")

    ranking1 = int(row1['ranking'].values[0])
    ranking2 = int(row2['ranking'].values[0])

    if ranking1 < ranking2:
        return team1
    elif ranking2 < ranking1:
        return team2
    else:
        return "Draw ranking"

In [32]:
def training_model(model_name, dataframe, target_col):
    """
    Entrena un modelo de Random Forest o XGBoost según el nombre indicado.

    Args:
        nombre_modelo (str): 'random_forest' o 'xgboost'
        dataframe (pd.DataFrame): Dataset completo con variables predictoras + target
        target_col (str): Nombre de la columna objetivo

    Returns:
        Modelo entrenado
    """
    if model_name not in ['random_forest', 'xgboost']:
        raise ValueError("Model not supported. Use 'random_forest' or 'xgboost'.")

    # Separar X e y
    X = dataframe.drop(columns=[target_col])
    y = dataframe[target_col]

    # Inicializar y entrenar el modelo
    if model_name == 'random_forest':
        model = RandomForestClassifier(n_estimators=1000, random_state=42)
    elif model_name == 'xgboost':
        model = XGBClassifier(n_estimators=1000, random_state=42, objective="reg:squarederror")

    model.fit(X, y)
    return model, X.columns.tolist()

In [33]:
def simular_serie_playoffs(
    modelo, dataset_home, dataset_away, tipo_dataset, equipo1, equipo2, obtener_localia_func,
    target_col="team_home_wins", csv_localias=None, model_features=None
):
    """
    Simula una serie de playoffs entre dos equipos, combinando estadísticas en base a localía.

    Returns:
        Nombre del equipo que gana la serie
    """
    """
    Simula una serie de playoffs entre dos equipos respetando el orden real de localía.

    Args:
        modelo: modelo entrenado (con predict_proba)
        dataset: DataFrame con estadísticas base (una fila por equipo)
        tipo_dataset: 'teams' o 'players'
        equipo1, equipo2: nombres de los equipos
        obtener_localia_func: función que devuelve quién arranca de local
        target_col: nombre de la variable objetivo

    Returns:
        Nombre del equipo que gana la serie
    """
    equipo_local = obtener_localia_func(csv_localias, equipo1, equipo2)
    equipo_visitante = equipo2 if equipo1 == equipo_local else equipo1

    partidos_localia = [equipo_local, equipo_local, equipo_visitante, equipo_visitante,
                        equipo_local, equipo_visitante, equipo_local]

    wins = {equipo_local: 0, equipo_visitante: 0}

    for local in partidos_localia:
        visitante = equipo_visitante if local == equipo_local else equipo_local

        fila_local = dataset_home[dataset_home["team"] == local].iloc[0]
        fila_visit = dataset_away[dataset_away["team"] == visitante].iloc[0]

        nueva_fila = {}

        if tipo_dataset == "teams":
            for col in fila_local.index:
                if col.startswith("team_") and col != "team":
                    nueva_fila["team_home_" + col[5:]] = fila_local[col]
            for col in fila_visit.index:
                if col.startswith("team_") and col != "team":
                    nueva_fila["team_away_" + col[5:]] = fila_visit[col]

        elif tipo_dataset == "players":
            for col in fila_local.index:
                if col != "team" and not col.startswith("th_") and not col.startswith("ta_"):
                    nueva_fila["th_" + col] = fila_local[col]
            for col in fila_visit.index:
                if col != "team" and not col.startswith("th_") and not col.startswith("ta_"):
                    nueva_fila["ta_" + col] = fila_visit[col]

        else:
            raise ValueError("tipo_dataset debe ser 'teams' o 'players'")

        X_test = pd.DataFrame([nueva_fila])

        if "team" in X_test.columns:
            X_test = X_test.drop(columns=["team"])
        
        X_test = X_test.reindex(columns=model_features)

        missing = set(model_features) - set(X_test.columns)
        if missing:
            raise ValueError(f"Faltan columnas necesarias en el dataset: {missing}")

        prob = modelo.predict_proba(X_test)[0, 1]
        ganador = local if prob >= 0.5 else visitante
        wins[ganador] += 1

        if wins[ganador] == 4:
            return ganador

    # Si llegan a 7 partidos sin uno con 4 victorias, decidir por mayoría
    return equipo_local if wins[equipo_local] > wins[equipo_visitante] else equipo_visitante

In [34]:
def simular_playoffs_completos(modelo, dataset_home, dataset_away, tipo_dataset,
                               csv_localias, target_col, obtener_localia_func,
                               model_features=None):
    """
    Simula todos los playoffs de la NBA y devuelve la lista ordenada de equipos:
    1. Por ronda alcanzada (1 campeón, 2 subcampeón, etc.)
    2. Por victorias en temporada regular
    3. Por promedio de puntos
    """
    df_localias = pd.read_csv(csv_localias)
    conferencias = ['east', 'west']

    # Diccionario para almacenar eliminaciones y ganador
    ranking_dict = {team: None for team in df_localias['team'].values}

    for conf in conferencias:
        equipos_conf = df_localias[df_localias['conference'] == conf].sort_values(by='ranking')
        sembrados = list(equipos_conf['team'].values)

        # Primera ronda
        emparejamientos = [
            (sembrados[0], sembrados[7]),
            (sembrados[3], sembrados[4]),
            (sembrados[2], sembrados[5]),
            (sembrados[1], sembrados[6])
        ]
        ronda_actual = []
        for eq1, eq2 in emparejamientos:
            ganador = simular_serie_playoffs(modelo, dataset_home, dataset_away,
                                             tipo_dataset, eq1, eq2, obtener_localia_func,
                                             target_col, csv_localias, model_features)
            perdedor = eq1 if ganador != eq1 else eq2
            ranking_dict[perdedor] = 15  # Eliminados en 1ra ronda
            ronda_actual.append(ganador)

        # Semifinales
        semis = [
            (ronda_actual[0], ronda_actual[1]),
            (ronda_actual[2], ronda_actual[3])
        ]
        ronda_semis = []
        for eq1, eq2 in semis:
            ganador = simular_serie_playoffs(modelo, dataset_home, dataset_away,
                                             tipo_dataset, eq1, eq2, obtener_localia_func,
                                             target_col, csv_localias, model_features)
            perdedor = eq1 if ganador != eq1 else eq2
            ranking_dict[perdedor] = 7  # Eliminados en 2da ronda
            ronda_semis.append(ganador)

        # Final de conferencia
        final_conf = (ronda_semis[0], ronda_semis[1])
        ganador_conf = simular_serie_playoffs(modelo, dataset_home, dataset_away,
                                              tipo_dataset, final_conf[0], final_conf[1],
                                              obtener_localia_func, target_col, csv_localias, model_features)
        perdedor_conf = final_conf[0] if ganador_conf != final_conf[0] else final_conf[1]
        ranking_dict[perdedor_conf] = 3  # Finalistas conferencia
        if conf == 'east':
            ganador_este = ganador_conf
        else:
            ganador_oeste = ganador_conf

    # Final NBA
    ganador_final = simular_serie_playoffs(modelo, dataset_home, dataset_away,
                                           tipo_dataset, ganador_este, ganador_oeste,
                                           obtener_localia_func, target_col, csv_localias, model_features)
    perdedor_final = ganador_este if ganador_final != ganador_este else ganador_oeste
    ranking_dict[perdedor_final] = 2  # Subcampeón
    ranking_dict[ganador_final] = 1   # Campeón

    # Unir con stats de temporada regular (wins y puntos)
    df_ranking = pd.DataFrame([
        {
            "equipo": equipo,
            "ronda": ronda,
            "wins_reg": df_localias.loc[df_localias["team"] == equipo, "wins_reg"].values[0],
            "points_avg": df_localias.loc[df_localias["team"] == equipo, "points_avg"].values[0]
        }
        for equipo, ronda in ranking_dict.items()
    ])

    # Ordenar según ronda > wins_reg > points_avg
    df_ranking.sort_values(
        by=["ronda", "wins_reg", "points_avg"],
        ascending=[True, False, False],
        inplace=True
    )

    # Devolver solo la lista ordenada
    return df_ranking["equipo"].tolist()

In [35]:
def comparar_brackets_booleano(bracket_real_csv, bracket_simulado_csv):
    """
    Compara dos brackets (real y simulado), ambos desde archivos CSV.

    Args:
        bracket_real_csv: path al archivo CSV con columnas ['equipo1', 'equipo2', 'ganador']
        bracket_simulado_csv: path al archivo CSV con columnas ['equipo1', 'equipo2', 'ganador']

    Returns:
        dict con:
            - lista de aciertos booleanos
            - MAE
            - MSE
            - aciertos totales
            - porcentaje de aciertos
    """
    sim_df = pd.read_csv(bracket_simulado_csv)
    real_df = pd.read_csv(bracket_real_csv)

    # Nos aseguramos de que ambas listas tengan los mismos equipos
    equipos_sim = set(sim_df["team"])
    equipos_real = set(real_df["team"])
    if equipos_sim != equipos_real:
        raise ValueError("Los equipos no coinciden entre el ranking real y el simulado")

    # Asignar posición de ranking (1 = mejor)
    sim_ranks = {team: rank+1 for rank, team in enumerate(sim_df["team"])}
    real_ranks = {team: rank+1 for rank, team in enumerate(real_df["team"])}

    # Crear vectores alineados
    equipos_ordenados = list(real_df["team"])  # usamos el orden del real
    sim_vector = [sim_ranks[e] for e in equipos_ordenados]
    real_vector = [real_ranks[e] for e in equipos_ordenados]

    # Calcular Pearson
    pearson_corr, _ = pearsonr(sim_vector, real_vector)

    return {
        "pearson": pearson_corr,
        "ranking_simulado": sim_ranks,
        "ranking_real": real_ranks
    }

# Players dataset, without dimentional reduction

In [36]:
df = pd.read_csv(f"final_data/season_players_dataset_{year}/season_players_dataset_with_context_{year}.csv")
df = df.drop(['game_id'], axis=1)
df_prueba_home = pd.read_csv(f"final_data/season_players_dataset_{year}/test_dataset/test_dataset_home_{year}.csv")
df_prueba_away = pd.read_csv(f"final_data/season_players_dataset_{year}/test_dataset/test_dataset_away_{year}.csv")

In [37]:
rf_model, model_features_rf = training_model('random_forest', df, "team_home_wins")
xgb_model, model_features_xgb = training_model('xgboost', df, "team_home_wins")

In [38]:
registro = simular_playoffs_completos(rf_model, df_prueba_home, df_prueba_away, 'players', csv_localias, "team_home_wins", get_home_priority, model_features_rf) # 'teams'

df_registro = pd.DataFrame(registro, columns=['team'])
path_bracket_simulated = "final_data/bracket_simulated.csv"
df_registro.to_csv(path_bracket_simulated, index=False)

In [39]:
metrics_rf = comparar_brackets_booleano(bracket_real_csv, path_bracket_simulated)
metrics_rf

{'pearson': np.float64(0.46470588235294125),
 'ranking_simulado': {'OKC': 1,
  'NYK': 2,
  'DEN': 3,
  'CLE': 4,
  'MIN': 5,
  'LAC': 6,
  'MIL': 7,
  'MIA': 8,
  'BOS': 9,
  'DAL': 10,
  'PHO': 11,
  'NOP': 12,
  'IND': 13,
  'LAL': 14,
  'PHI': 15,
  'ORL': 16},
 'ranking_real': {'BOS': 1,
  'DAL': 2,
  'MIN': 3,
  'IND': 4,
  'OKC': 5,
  'DEN': 6,
  'NYK': 7,
  'CLE': 8,
  'LAC': 9,
  'MIL': 10,
  'PHO': 11,
  'NOP': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16}}

In [40]:
registro = simular_playoffs_completos(xgb_model, df_prueba_home, df_prueba_away, 'players', csv_localias, "team_home_wins", get_home_priority, model_features_xgb) # 'teams'

df_registro = pd.DataFrame(registro, columns=['team'])
path_bracket_simulated = "final_data/bracket_simulated.csv"
df_registro.to_csv(path_bracket_simulated, index=False)

In [41]:
metrics_xgb = comparar_brackets_booleano(bracket_real_csv, path_bracket_simulated)
metrics_xgb

{'pearson': np.float64(0.7588235294117647),
 'ranking_simulado': {'BOS': 1,
  'OKC': 2,
  'DEN': 3,
  'NYK': 4,
  'MIN': 5,
  'LAC': 6,
  'MIL': 7,
  'CLE': 8,
  'DAL': 9,
  'PHO': 10,
  'NOP': 11,
  'IND': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16},
 'ranking_real': {'BOS': 1,
  'DAL': 2,
  'MIN': 3,
  'IND': 4,
  'OKC': 5,
  'DEN': 6,
  'NYK': 7,
  'CLE': 8,
  'LAC': 9,
  'MIL': 10,
  'PHO': 11,
  'NOP': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16}}

# Teams dataset, without dimentional reduction

In [42]:
df = pd.read_csv(f"final_data/season_teams_dataset_{year}/season_teams_dataset_with_context_{year}.csv")
df = df.drop(['game_id'], axis=1)
df_prueba_home = pd.read_csv(f"final_data/season_teams_dataset_{year}/test_dataset/test_dataset_home_{year}.csv")
df_prueba_away = pd.read_csv(f"final_data/season_teams_dataset_{year}/test_dataset/test_dataset_away_{year}.csv")

In [43]:
rf_model, model_features_rf = training_model('random_forest', df, "team_home_wins")
xgb_model, model_features_xgb = training_model('xgboost', df, "team_home_wins")

In [44]:
registro = simular_playoffs_completos(rf_model, df_prueba_home, df_prueba_away, 'teams', csv_localias, "team_home_wins", get_home_priority, model_features_rf) # 'teams'

df_registro = pd.DataFrame(registro, columns=['team'])
path_bracket_simulated = "final_data/bracket_simulated.csv"
df_registro.to_csv(path_bracket_simulated, index=False)

In [45]:
metrics_rf = comparar_brackets_booleano(bracket_real_csv, path_bracket_simulated)
metrics_rf

{'pearson': np.float64(0.7588235294117647),
 'ranking_simulado': {'BOS': 1,
  'OKC': 2,
  'DEN': 3,
  'NYK': 4,
  'MIN': 5,
  'LAC': 6,
  'MIL': 7,
  'CLE': 8,
  'DAL': 9,
  'PHO': 10,
  'NOP': 11,
  'IND': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16},
 'ranking_real': {'BOS': 1,
  'DAL': 2,
  'MIN': 3,
  'IND': 4,
  'OKC': 5,
  'DEN': 6,
  'NYK': 7,
  'CLE': 8,
  'LAC': 9,
  'MIL': 10,
  'PHO': 11,
  'NOP': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16}}

In [46]:
registro = simular_playoffs_completos(xgb_model, df_prueba_home, df_prueba_away, 'teams', csv_localias, "team_home_wins", get_home_priority, model_features_xgb) # 'teams'

df_registro = pd.DataFrame(registro, columns=['team'])
path_bracket_simulated = "final_data/bracket_simulated.csv"
df_registro.to_csv(path_bracket_simulated, index=False)

In [47]:
metrics_xgb = comparar_brackets_booleano(bracket_real_csv, path_bracket_simulated)
metrics_xgb

{'pearson': np.float64(-0.4205882352941177),
 'ranking_simulado': {'MIA': 1,
  'NOP': 2,
  'LAL': 3,
  'PHI': 4,
  'DAL': 5,
  'PHO': 6,
  'IND': 7,
  'ORL': 8,
  'BOS': 9,
  'OKC': 10,
  'DEN': 11,
  'MIN': 12,
  'LAC': 13,
  'NYK': 14,
  'MIL': 15,
  'CLE': 16},
 'ranking_real': {'BOS': 1,
  'DAL': 2,
  'MIN': 3,
  'IND': 4,
  'OKC': 5,
  'DEN': 6,
  'NYK': 7,
  'CLE': 8,
  'LAC': 9,
  'MIL': 10,
  'PHO': 11,
  'NOP': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16}}

# Players dataset, with dimentional reduction

### All components

In [48]:
df = pd.read_csv(f"final_data/season_players_dataset_{year}_reduced/season_players_dataset_with_context_{year}.csv")
df = df.drop(['game_id'], axis=1)
df_prueba_home = pd.read_csv(f"final_data/season_players_dataset_{year}_reduced/test_dataset/test_dataset_home_{year}.csv")
df_prueba_away = pd.read_csv(f"final_data/season_players_dataset_{year}_reduced/test_dataset/test_dataset_away_{year}.csv")

In [49]:
rf_model, model_features_rf = training_model('random_forest', df, "team_home_wins")
xgb_model, model_features_xgb = training_model('xgboost', df, "team_home_wins")

In [50]:
registro = simular_playoffs_completos(rf_model, df_prueba_home, df_prueba_away, 'players', csv_localias, "team_home_wins", get_home_priority, model_features_rf) # 'teams'

df_registro = pd.DataFrame(registro, columns=['team'])
path_bracket_simulated = "final_data/bracket_simulated.csv"
df_registro.to_csv(path_bracket_simulated, index=False)

In [51]:
metrics_rf = comparar_brackets_booleano(bracket_real_csv, path_bracket_simulated)
metrics_rf

{'pearson': np.float64(0.46470588235294125),
 'ranking_simulado': {'OKC': 1,
  'NYK': 2,
  'DEN': 3,
  'CLE': 4,
  'MIN': 5,
  'LAC': 6,
  'MIL': 7,
  'MIA': 8,
  'BOS': 9,
  'DAL': 10,
  'PHO': 11,
  'NOP': 12,
  'IND': 13,
  'LAL': 14,
  'PHI': 15,
  'ORL': 16},
 'ranking_real': {'BOS': 1,
  'DAL': 2,
  'MIN': 3,
  'IND': 4,
  'OKC': 5,
  'DEN': 6,
  'NYK': 7,
  'CLE': 8,
  'LAC': 9,
  'MIL': 10,
  'PHO': 11,
  'NOP': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16}}

In [52]:
registro = simular_playoffs_completos(xgb_model, df_prueba_home, df_prueba_away, 'players', csv_localias, "team_home_wins", get_home_priority, model_features_xgb) # 'teams'

df_registro = pd.DataFrame(registro, columns=['team'])
path_bracket_simulated = "final_data/bracket_simulated.csv"
df_registro.to_csv(path_bracket_simulated, index=False)

In [53]:
metrics_xgb = comparar_brackets_booleano(bracket_real_csv, path_bracket_simulated)
metrics_xgb

{'pearson': np.float64(0.46470588235294125),
 'ranking_simulado': {'OKC': 1,
  'NYK': 2,
  'DEN': 3,
  'CLE': 4,
  'MIN': 5,
  'LAC': 6,
  'MIL': 7,
  'MIA': 8,
  'BOS': 9,
  'DAL': 10,
  'PHO': 11,
  'NOP': 12,
  'IND': 13,
  'LAL': 14,
  'PHI': 15,
  'ORL': 16},
 'ranking_real': {'BOS': 1,
  'DAL': 2,
  'MIN': 3,
  'IND': 4,
  'OKC': 5,
  'DEN': 6,
  'NYK': 7,
  'CLE': 8,
  'LAC': 9,
  'MIL': 10,
  'PHO': 11,
  'NOP': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16}}

### Three firsts components

In [54]:
filter_cols = [col for col in df.columns if not re.search(r'_(\d+)$', col) or int(re.search(r'_(\d+)$', col).group(1)) <= 3]
df_first_second_third_components = df[filter_cols]

filter_cols = [col for col in df_prueba_home.columns if not re.search(r'_(\d+)$', col) or int(re.search(r'_(\d+)$', col).group(1)) <= 3]
df_prueba_home_first_second_third_components = df_prueba_home[filter_cols]

filter_cols = [col for col in df_prueba_away.columns if not re.search(r'_(\d+)$', col) or int(re.search(r'_(\d+)$', col).group(1)) <= 3]
df_prueba_away_first_second_third_components = df_prueba_away[filter_cols]

In [55]:
rf_model, model_features_rf = training_model('random_forest', df_first_second_third_components, "team_home_wins")
xgb_model, model_features_xgb = training_model('xgboost', df_first_second_third_components, "team_home_wins")

In [56]:
registro = simular_playoffs_completos(rf_model, df_prueba_home_first_second_third_components, df_prueba_away_first_second_third_components, 
                                      'players', csv_localias, "team_home_wins", get_home_priority, model_features_rf) # 'teams'

df_registro = pd.DataFrame(registro, columns=['team'])
path_bracket_simulated = "final_data/bracket_simulated.csv"
df_registro.to_csv(path_bracket_simulated, index=False)

In [57]:
metrics_rf = comparar_brackets_booleano(bracket_real_csv, path_bracket_simulated)
metrics_rf

{'pearson': np.float64(0.46470588235294125),
 'ranking_simulado': {'OKC': 1,
  'NYK': 2,
  'DEN': 3,
  'CLE': 4,
  'MIN': 5,
  'LAC': 6,
  'MIL': 7,
  'MIA': 8,
  'BOS': 9,
  'DAL': 10,
  'PHO': 11,
  'NOP': 12,
  'IND': 13,
  'LAL': 14,
  'PHI': 15,
  'ORL': 16},
 'ranking_real': {'BOS': 1,
  'DAL': 2,
  'MIN': 3,
  'IND': 4,
  'OKC': 5,
  'DEN': 6,
  'NYK': 7,
  'CLE': 8,
  'LAC': 9,
  'MIL': 10,
  'PHO': 11,
  'NOP': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16}}

In [59]:
registro = simular_playoffs_completos(xgb_model, df_prueba_home_first_second_third_components, df_prueba_away_first_second_third_components, 
                                      'players', csv_localias, "team_home_wins", get_home_priority, model_features_xgb) # 'teams'

df_registro = pd.DataFrame(registro, columns=['team'])
path_bracket_simulated = "final_data/bracket_simulated.csv"
df_registro.to_csv(path_bracket_simulated, index=False)

In [60]:
metrics_xgb = comparar_brackets_booleano(bracket_real_csv, path_bracket_simulated)
metrics_xgb

{'pearson': np.float64(0.46470588235294125),
 'ranking_simulado': {'OKC': 1,
  'NYK': 2,
  'DEN': 3,
  'CLE': 4,
  'MIN': 5,
  'LAC': 6,
  'MIL': 7,
  'MIA': 8,
  'BOS': 9,
  'DAL': 10,
  'PHO': 11,
  'NOP': 12,
  'IND': 13,
  'LAL': 14,
  'PHI': 15,
  'ORL': 16},
 'ranking_real': {'BOS': 1,
  'DAL': 2,
  'MIN': 3,
  'IND': 4,
  'OKC': 5,
  'DEN': 6,
  'NYK': 7,
  'CLE': 8,
  'LAC': 9,
  'MIL': 10,
  'PHO': 11,
  'NOP': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16}}

### Two firsts components

In [61]:
filter_cols = [col for col in df.columns if not re.search(r'_(\d+)$', col) or int(re.search(r'_(\d+)$', col).group(1)) <= 2]
df_first_second_components = df[filter_cols]

filter_cols = [col for col in df_prueba_home.columns if not re.search(r'_(\d+)$', col) or int(re.search(r'_(\d+)$', col).group(1)) <= 2]
df_prueba_home_first_second_components = df_prueba_home[filter_cols]

filter_cols = [col for col in df_prueba_away.columns if not re.search(r'_(\d+)$', col) or int(re.search(r'_(\d+)$', col).group(1)) <= 2]
df_prueba_away_first_second_components = df_prueba_away[filter_cols]

In [62]:
rf_model, model_features_rf = training_model('random_forest', df_first_second_components, "team_home_wins")
xgb_model, model_features_xgb = training_model('xgboost', df_first_second_components, "team_home_wins")

In [63]:
registro = simular_playoffs_completos(rf_model, df_prueba_home_first_second_components, df_prueba_away_first_second_components, 
                                      'players', csv_localias, "team_home_wins", get_home_priority, model_features_rf) # 'teams'

df_registro = pd.DataFrame(registro, columns=['team'])
path_bracket_simulated = "final_data/bracket_simulated.csv"
df_registro.to_csv(path_bracket_simulated, index=False)

In [64]:
metrics_rf = comparar_brackets_booleano(bracket_real_csv, path_bracket_simulated)
metrics_rf

{'pearson': np.float64(0.7588235294117647),
 'ranking_simulado': {'BOS': 1,
  'OKC': 2,
  'DEN': 3,
  'NYK': 4,
  'MIN': 5,
  'LAC': 6,
  'MIL': 7,
  'CLE': 8,
  'DAL': 9,
  'PHO': 10,
  'NOP': 11,
  'IND': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16},
 'ranking_real': {'BOS': 1,
  'DAL': 2,
  'MIN': 3,
  'IND': 4,
  'OKC': 5,
  'DEN': 6,
  'NYK': 7,
  'CLE': 8,
  'LAC': 9,
  'MIL': 10,
  'PHO': 11,
  'NOP': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16}}

In [65]:
registro = simular_playoffs_completos(xgb_model, df_prueba_home_first_second_components, df_prueba_away_first_second_components, 
                                      'players', csv_localias, "team_home_wins", get_home_priority, model_features_xgb) # 'teams'

df_registro = pd.DataFrame(registro, columns=['team'])
path_bracket_simulated = "final_data/bracket_simulated.csv"
df_registro.to_csv(path_bracket_simulated, index=False)

In [66]:
metrics_xgb = comparar_brackets_booleano(bracket_real_csv, path_bracket_simulated)
metrics_xgb

{'pearson': np.float64(0.8441176470588235),
 'ranking_simulado': {'BOS': 1,
  'OKC': 2,
  'DEN': 3,
  'NYK': 4,
  'MIN': 5,
  'LAC': 6,
  'CLE': 7,
  'IND': 8,
  'DAL': 9,
  'MIL': 10,
  'PHO': 11,
  'NOP': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16},
 'ranking_real': {'BOS': 1,
  'DAL': 2,
  'MIN': 3,
  'IND': 4,
  'OKC': 5,
  'DEN': 6,
  'NYK': 7,
  'CLE': 8,
  'LAC': 9,
  'MIL': 10,
  'PHO': 11,
  'NOP': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16}}

### First component

In [67]:
filter_cols = [col for col in df.columns if not re.search(r'_(\d+)$', col) or int(re.search(r'_(\d+)$', col).group(1)) <= 1]
df_first_components = df[filter_cols]

filter_cols = [col for col in df_prueba_home.columns if not re.search(r'_(\d+)$', col) or int(re.search(r'_(\d+)$', col).group(1)) <= 1]
df_prueba_home_first_components = df_prueba_home[filter_cols]

filter_cols = [col for col in df_prueba_away.columns if not re.search(r'_(\d+)$', col) or int(re.search(r'_(\d+)$', col).group(1)) <= 1]
df_prueba_away_first_components = df_prueba_away[filter_cols]

In [68]:
rf_model, model_features_rf = training_model('random_forest', df_first_components, "team_home_wins")
xgb_model, model_features_xgb = training_model('xgboost', df_first_components, "team_home_wins")

In [69]:
registro = simular_playoffs_completos(rf_model, df_prueba_home_first_components, df_prueba_away_first_components,
                                       'players', csv_localias, "team_home_wins", get_home_priority, model_features_rf) # 'teams'

df_registro = pd.DataFrame(registro, columns=['team'])
path_bracket_simulated = "final_data/bracket_simulated.csv"
df_registro.to_csv(path_bracket_simulated, index=False)

In [70]:
metrics_rf = comparar_brackets_booleano(bracket_real_csv, path_bracket_simulated)
metrics_rf

{'pearson': np.float64(0.7588235294117647),
 'ranking_simulado': {'BOS': 1,
  'OKC': 2,
  'DEN': 3,
  'NYK': 4,
  'MIN': 5,
  'LAC': 6,
  'MIL': 7,
  'CLE': 8,
  'DAL': 9,
  'PHO': 10,
  'NOP': 11,
  'IND': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16},
 'ranking_real': {'BOS': 1,
  'DAL': 2,
  'MIN': 3,
  'IND': 4,
  'OKC': 5,
  'DEN': 6,
  'NYK': 7,
  'CLE': 8,
  'LAC': 9,
  'MIL': 10,
  'PHO': 11,
  'NOP': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16}}

In [71]:
registro = simular_playoffs_completos(xgb_model, df_prueba_home_first_components, df_prueba_away_first_components,
                                    'players', csv_localias, "team_home_wins", get_home_priority, model_features_xgb) # 'teams'

df_registro = pd.DataFrame(registro, columns=['team'])
path_bracket_simulated = "final_data/bracket_simulated.csv"
df_registro.to_csv(path_bracket_simulated, index=False)

In [72]:
metrics_xgb = comparar_brackets_booleano(bracket_real_csv, path_bracket_simulated)
metrics_xgb

{'pearson': np.float64(0.7588235294117647),
 'ranking_simulado': {'BOS': 1,
  'OKC': 2,
  'DEN': 3,
  'NYK': 4,
  'MIN': 5,
  'LAC': 6,
  'MIL': 7,
  'CLE': 8,
  'DAL': 9,
  'PHO': 10,
  'NOP': 11,
  'IND': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16},
 'ranking_real': {'BOS': 1,
  'DAL': 2,
  'MIN': 3,
  'IND': 4,
  'OKC': 5,
  'DEN': 6,
  'NYK': 7,
  'CLE': 8,
  'LAC': 9,
  'MIL': 10,
  'PHO': 11,
  'NOP': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16}}

# Teams dataset, with dimentional reduction

### All components

In [73]:
df = pd.read_csv(f"final_data/season_teams_dataset_{year}_reduced/season_teams_dataset_with_context_{year}.csv")
df = df.drop(['game_id'], axis=1)
df = df.select_dtypes(exclude=['object', 'string'])
df_prueba_home = pd.read_csv(f"final_data/season_teams_dataset_{year}_reduced/test_dataset/test_dataset_home_{year}.csv")
df_prueba_away = pd.read_csv(f"final_data/season_teams_dataset_{year}_reduced/test_dataset/test_dataset_away_{year}.csv")

In [74]:
rf_model, model_features_rf = training_model('random_forest', df, "team_home_wins")
xgb_model, model_features_xgb = training_model('xgboost', df, "team_home_wins")

In [75]:
registro = simular_playoffs_completos(rf_model, df_prueba_home, df_prueba_away, 'teams', csv_localias, "team_home_wins", get_home_priority, model_features_rf) # 'teams'

df_registro = pd.DataFrame(registro, columns=['team'])
path_bracket_simulated = "final_data/bracket_simulated.csv"
df_registro.to_csv(path_bracket_simulated, index=False)

In [76]:
metrics_rf = comparar_brackets_booleano(bracket_real_csv, path_bracket_simulated)
metrics_rf

{'pearson': np.float64(-0.4205882352941177),
 'ranking_simulado': {'MIA': 1,
  'NOP': 2,
  'LAL': 3,
  'PHI': 4,
  'DAL': 5,
  'PHO': 6,
  'IND': 7,
  'ORL': 8,
  'BOS': 9,
  'OKC': 10,
  'DEN': 11,
  'MIN': 12,
  'LAC': 13,
  'NYK': 14,
  'MIL': 15,
  'CLE': 16},
 'ranking_real': {'BOS': 1,
  'DAL': 2,
  'MIN': 3,
  'IND': 4,
  'OKC': 5,
  'DEN': 6,
  'NYK': 7,
  'CLE': 8,
  'LAC': 9,
  'MIL': 10,
  'PHO': 11,
  'NOP': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16}}

In [77]:
registro = simular_playoffs_completos(xgb_model, df_prueba_home, df_prueba_away, 'teams', csv_localias, "team_home_wins", get_home_priority, model_features_xgb) # 'teams'

df_registro = pd.DataFrame(registro, columns=['team'])
path_bracket_simulated = "final_data/bracket_simulated.csv"
df_registro.to_csv(path_bracket_simulated, index=False)

In [78]:
metrics_xgb = comparar_brackets_booleano(bracket_real_csv, path_bracket_simulated)
metrics_xgb

{'pearson': np.float64(0.7588235294117647),
 'ranking_simulado': {'BOS': 1,
  'OKC': 2,
  'DEN': 3,
  'NYK': 4,
  'MIN': 5,
  'LAC': 6,
  'MIL': 7,
  'CLE': 8,
  'DAL': 9,
  'PHO': 10,
  'NOP': 11,
  'IND': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16},
 'ranking_real': {'BOS': 1,
  'DAL': 2,
  'MIN': 3,
  'IND': 4,
  'OKC': 5,
  'DEN': 6,
  'NYK': 7,
  'CLE': 8,
  'LAC': 9,
  'MIL': 10,
  'PHO': 11,
  'NOP': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16}}

### Three firsts components

In [79]:
filter_cols = [col for col in df.columns if not re.search(r'_(\d+)$', col) or int(re.search(r'_(\d+)$', col).group(1)) <= 3]
df_first_second_third_components = df[filter_cols]

filter_cols = [col for col in df_prueba_home.columns if not re.search(r'_(\d+)$', col) or int(re.search(r'_(\d+)$', col).group(1)) <= 3]
df_prueba_home_first_second_third_components = df_prueba_home[filter_cols]

filter_cols = [col for col in df_prueba_away.columns if not re.search(r'_(\d+)$', col) or int(re.search(r'_(\d+)$', col).group(1)) <= 3]
df_prueba_away_first_second_third_components = df_prueba_away[filter_cols]

In [80]:
rf_model, model_features_rf = training_model('random_forest', df_first_second_third_components, "team_home_wins")
xgb_model, model_features_xgb = training_model('xgboost', df_first_second_third_components, "team_home_wins")

In [81]:
registro = simular_playoffs_completos(rf_model, df_prueba_home_first_second_third_components, df_prueba_away_first_second_third_components, 
                                      'teams', csv_localias, "team_home_wins", get_home_priority, model_features_rf) # 'teams'

df_registro = pd.DataFrame(registro, columns=['team'])
path_bracket_simulated = "final_data/bracket_simulated.csv"
df_registro.to_csv(path_bracket_simulated, index=False)

In [82]:
metrics_rf = comparar_brackets_booleano(bracket_real_csv, path_bracket_simulated)
metrics_rf

{'pearson': np.float64(-0.4205882352941177),
 'ranking_simulado': {'MIA': 1,
  'NOP': 2,
  'LAL': 3,
  'PHI': 4,
  'DAL': 5,
  'PHO': 6,
  'IND': 7,
  'ORL': 8,
  'BOS': 9,
  'OKC': 10,
  'DEN': 11,
  'MIN': 12,
  'LAC': 13,
  'NYK': 14,
  'MIL': 15,
  'CLE': 16},
 'ranking_real': {'BOS': 1,
  'DAL': 2,
  'MIN': 3,
  'IND': 4,
  'OKC': 5,
  'DEN': 6,
  'NYK': 7,
  'CLE': 8,
  'LAC': 9,
  'MIL': 10,
  'PHO': 11,
  'NOP': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16}}

In [83]:
registro = simular_playoffs_completos(xgb_model, df_prueba_home_first_second_third_components, df_prueba_away_first_second_third_components, 
                                      'teams', csv_localias, "team_home_wins", get_home_priority, model_features_xgb) # 'teams'

df_registro = pd.DataFrame(registro, columns=['team'])
path_bracket_simulated = "final_data/bracket_simulated.csv"
df_registro.to_csv(path_bracket_simulated, index=False)

In [84]:
metrics_xgb = comparar_brackets_booleano(bracket_real_csv, path_bracket_simulated)
metrics_xgb

{'pearson': np.float64(0.7588235294117647),
 'ranking_simulado': {'BOS': 1,
  'OKC': 2,
  'DEN': 3,
  'NYK': 4,
  'MIN': 5,
  'LAC': 6,
  'MIL': 7,
  'CLE': 8,
  'DAL': 9,
  'PHO': 10,
  'NOP': 11,
  'IND': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16},
 'ranking_real': {'BOS': 1,
  'DAL': 2,
  'MIN': 3,
  'IND': 4,
  'OKC': 5,
  'DEN': 6,
  'NYK': 7,
  'CLE': 8,
  'LAC': 9,
  'MIL': 10,
  'PHO': 11,
  'NOP': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16}}

### Two firsts components

In [85]:
filter_cols = [col for col in df.columns if not re.search(r'_(\d+)$', col) or int(re.search(r'_(\d+)$', col).group(1)) <= 2]
df_first_second_components = df[filter_cols]

filter_cols = [col for col in df_prueba_home.columns if not re.search(r'_(\d+)$', col) or int(re.search(r'_(\d+)$', col).group(1)) <= 2]
df_prueba_home_first_second_components = df_prueba_home[filter_cols]

filter_cols = [col for col in df_prueba_away.columns if not re.search(r'_(\d+)$', col) or int(re.search(r'_(\d+)$', col).group(1)) <= 2]
df_prueba_away_first_second_components = df_prueba_away[filter_cols]

In [86]:
rf_model, model_features_rf = training_model('random_forest', df_first_second_components, "team_home_wins")
xgb_model, model_features_xgb = training_model('xgboost', df_first_second_components, "team_home_wins")

In [87]:
registro = simular_playoffs_completos(rf_model, df_prueba_home_first_second_components, df_prueba_away_first_second_components, 
                                      'teams', csv_localias, "team_home_wins", get_home_priority, model_features_rf) # 'teams'

df_registro = pd.DataFrame(registro, columns=['team'])
path_bracket_simulated = "final_data/bracket_simulated.csv"
df_registro.to_csv(path_bracket_simulated, index=False)

In [88]:
metrics_rf = comparar_brackets_booleano(bracket_real_csv, path_bracket_simulated)
metrics_rf

{'pearson': np.float64(0.7588235294117647),
 'ranking_simulado': {'BOS': 1,
  'OKC': 2,
  'DEN': 3,
  'NYK': 4,
  'MIN': 5,
  'LAC': 6,
  'MIL': 7,
  'CLE': 8,
  'DAL': 9,
  'PHO': 10,
  'NOP': 11,
  'IND': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16},
 'ranking_real': {'BOS': 1,
  'DAL': 2,
  'MIN': 3,
  'IND': 4,
  'OKC': 5,
  'DEN': 6,
  'NYK': 7,
  'CLE': 8,
  'LAC': 9,
  'MIL': 10,
  'PHO': 11,
  'NOP': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16}}

In [89]:
registro = simular_playoffs_completos(xgb_model, df_prueba_home_first_second_components, df_prueba_away_first_second_components, 
                                      'teams', csv_localias, "team_home_wins", get_home_priority, model_features_xgb) # 'teams'

df_registro = pd.DataFrame(registro, columns=['team'])
path_bracket_simulated = "final_data/bracket_simulated.csv"
df_registro.to_csv(path_bracket_simulated, index=False)

In [90]:
metrics_xgb = comparar_brackets_booleano(bracket_real_csv, path_bracket_simulated)
metrics_xgb

{'pearson': np.float64(-0.4205882352941177),
 'ranking_simulado': {'MIA': 1,
  'NOP': 2,
  'LAL': 3,
  'PHI': 4,
  'DAL': 5,
  'PHO': 6,
  'IND': 7,
  'ORL': 8,
  'BOS': 9,
  'OKC': 10,
  'DEN': 11,
  'MIN': 12,
  'LAC': 13,
  'NYK': 14,
  'MIL': 15,
  'CLE': 16},
 'ranking_real': {'BOS': 1,
  'DAL': 2,
  'MIN': 3,
  'IND': 4,
  'OKC': 5,
  'DEN': 6,
  'NYK': 7,
  'CLE': 8,
  'LAC': 9,
  'MIL': 10,
  'PHO': 11,
  'NOP': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16}}

### First component

In [91]:
filter_cols = [col for col in df.columns if not re.search(r'_(\d+)$', col) or int(re.search(r'_(\d+)$', col).group(1)) <= 1]
df_first_components = df[filter_cols]

filter_cols = [col for col in df_prueba_home.columns if not re.search(r'_(\d+)$', col) or int(re.search(r'_(\d+)$', col).group(1)) <= 1]
df_prueba_home_first_components = df_prueba_home[filter_cols]

filter_cols = [col for col in df_prueba_away.columns if not re.search(r'_(\d+)$', col) or int(re.search(r'_(\d+)$', col).group(1)) <= 1]
df_prueba_away_first_components = df_prueba_away[filter_cols]

In [92]:
rf_model, model_features_rf = training_model('random_forest', df_first_components, "team_home_wins")
xgb_model, model_features_xgb = training_model('xgboost', df_first_components, "team_home_wins")

In [93]:
registro = simular_playoffs_completos(rf_model, df_prueba_home_first_components, df_prueba_away_first_components, 
                                      'teams', csv_localias, "team_home_wins", get_home_priority, model_features_rf) # 'teams'

df_registro = pd.DataFrame(registro, columns=['team'])
path_bracket_simulated = "final_data/bracket_simulated.csv"
df_registro.to_csv(path_bracket_simulated, index=False)

In [94]:
metrics_rf = comparar_brackets_booleano(bracket_real_csv, path_bracket_simulated)
metrics_rf

{'pearson': np.float64(-0.4205882352941177),
 'ranking_simulado': {'MIA': 1,
  'NOP': 2,
  'LAL': 3,
  'PHI': 4,
  'DAL': 5,
  'PHO': 6,
  'IND': 7,
  'ORL': 8,
  'BOS': 9,
  'OKC': 10,
  'DEN': 11,
  'MIN': 12,
  'LAC': 13,
  'NYK': 14,
  'MIL': 15,
  'CLE': 16},
 'ranking_real': {'BOS': 1,
  'DAL': 2,
  'MIN': 3,
  'IND': 4,
  'OKC': 5,
  'DEN': 6,
  'NYK': 7,
  'CLE': 8,
  'LAC': 9,
  'MIL': 10,
  'PHO': 11,
  'NOP': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16}}

In [95]:
registro = simular_playoffs_completos(xgb_model, df_prueba_home_first_components, df_prueba_away_first_components,
                                      'teams', csv_localias, "team_home_wins", get_home_priority, model_features_xgb) # 'teams'

df_registro = pd.DataFrame(registro, columns=['team'])
path_bracket_simulated = "final_data/bracket_simulated.csv"
df_registro.to_csv(path_bracket_simulated, index=False)

In [96]:
metrics_xgb = comparar_brackets_booleano(bracket_real_csv, path_bracket_simulated)
metrics_xgb

{'pearson': np.float64(0.7588235294117647),
 'ranking_simulado': {'BOS': 1,
  'OKC': 2,
  'DEN': 3,
  'NYK': 4,
  'MIN': 5,
  'LAC': 6,
  'MIL': 7,
  'CLE': 8,
  'DAL': 9,
  'PHO': 10,
  'NOP': 11,
  'IND': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16},
 'ranking_real': {'BOS': 1,
  'DAL': 2,
  'MIN': 3,
  'IND': 4,
  'OKC': 5,
  'DEN': 6,
  'NYK': 7,
  'CLE': 8,
  'LAC': 9,
  'MIL': 10,
  'PHO': 11,
  'NOP': 12,
  'LAL': 13,
  'PHI': 14,
  'ORL': 15,
  'MIA': 16}}